[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ODJ0930/basalt/blob/main/Basalt_DeepSeek41_Colab.ipynb)

# DeepSeek-V4.1-Flash on a Colab G4 (Basalt)

Runs DeepSeek-V4.1-Flash-1017 with [Basalt](https://github.com/jesdga95/basalt), an engine made for NVIDIA Blackwell, and
serves its OpenAI / Anthropic compatible API through a Cloudflare tunnel.

* **Runtime:** *Runtime → Change runtime type → G4* (RTX PRO 6000 Blackwell, 96 GB). Basalt needs compute capability 12.0, so H100 / A100 / L4 will not work.
* **Storage:** the pack is one ~242 GiB file and the G4 disk is ~236 GiB, so step 2 joins 220 GiB of disk and 30 GiB of RAM into one volume.
* Every session downloads the pack again (~17 min at ~260 MB/s) and rebuilds the engine (~2 min) in the background; loading then takes ~7 min.
* Measured on a G4: 129.5 GiB loaded (76 GiB VRAM + 53 GiB RAM, every expert resident), first token ~1.2 s, ~53 tok/s.

### 1. Check the runtime

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
assert ", 12.0," in gpu, "Basalt needs a Blackwell card (compute capability 12.0): pick the G4 runtime"
!free -g | head -2; df -h / | tail -1

### 2. Get Basalt and make the disk + RAM volume

In [ ]:
!git clone -q --depth 1 https://github.com/ODJ0930/basalt.git /content/basalt 2>/dev/null || git -C /content/basalt pull -q
!bash /content/basalt/colab/storage.sh

### 3. Download the pack and build Basalt (background)

In [ ]:
PACK = "DeepSeek-V4.1-Flash-1017-UD-IQ1_M.basalt"  # @param ["DeepSeek-V4.1-Flash-1017-UD-IQ1_M.basalt"]
import os, subprocess, time
from pathlib import Path

def background(script, log):
    return subprocess.Popen(["bash", script], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                            stdin=subprocess.DEVNULL, start_new_session=True, env={**os.environ, "PACK": PACK})

if not Path(f"/mnt/models/{PACK}").exists():
    download = background("/content/basalt/colab/download.sh", "/content/download.log")
build = background("/content/basalt/colab/build.sh", "/content/build.log")

def done(log, mark):
    return Path(log).exists() and mark in Path(log).read_text(errors="replace")

start = time.time()
while not (done("/content/build.log", "BUILD_DONE") and
           (Path(f"/mnt/models/{PACK}").exists() or done("/content/download.log", "DOWNLOAD_DONE"))):
    for proc, log in ((build, "/content/build.log"),) + (((download, "/content/download.log"),) if "download" in globals() else ()):
        if proc.poll() not in (None, 0):
            !tail -20 {log}
            raise RuntimeError(f"{log}: the step failed")
    used = subprocess.run(["df", "-BG", "--output=used", "/mnt/models"], capture_output=True, text=True).stdout.split()[-1]
    print(f"[{(time.time() - start) / 60:4.1f} min] downloaded {used} of ~242G, build "
          f"{'done' if done('/content/build.log', 'BUILD_DONE') else 'running'}", flush=True)
    time.sleep(60)
print("pack and engine ready")

### 4. Start the server and wait for the model to load

In [ ]:
import json, urllib.request
CONTEXT = 32768  # @param {type:"integer"}
!PACK=/mnt/models/{PACK} CONTEXT={CONTEXT} bash /content/basalt/colab/serve.sh
API_KEY = Path("/content/basalt_api_key.txt").read_text().strip()

def health():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8090/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

for i in range(240):  # up to 40 minutes
    h = health()
    if h and h.get("loaded"):
        print("model loaded:", h.get("model"), "context", h.get("max_context"))
        break
    if subprocess.run(["pgrep", "-f", "[s]erve.server"], capture_output=True).returncode != 0:
        !tail -30 /content/basalt-server.log
        raise RuntimeError("the server stopped")
    if i % 3 == 0:
        print(f"[{i / 6:4.1f} min] loading {h.get('percent') if h else '...'}%", flush=True)
    time.sleep(10)
else:
    !tail -30 /content/basalt-server.log
    raise RuntimeError("the model did not load in 40 minutes")
!grep -i "expert homes\|RAM floor\|drafter" /content/basalt-server.log | tail -5

### 5. Quick test

In [ ]:
# streams one request; prints time to first token and tokens/s (measured on a G4: ~1.2 s and ~53 tok/s)
!python3 /content/basalt/colab/test.py

### 6. Public URL (Cloudflare quick tunnel)

Keep the API key private: anyone with the URL and the key can use this runtime.

In [ ]:
!bash /content/basalt/colab/tunnel.sh

### 7. Keep the runtime alive

Leave this running (and the tab open) while you use the API.

In [ ]:
!python3 /content/basalt/colab/keepalive.py